# 03. Microsoft GraphRAG: 여섯 원문에서 자동 인덱스까지

2번 노트북에서는 사람이 D1–D3의 관계를 직접 골랐습니다. 여기서는 [Microsoft GraphRAG](https://microsoft.github.io/graphrag/get_started/)가 문서를 읽고 **TextUnit, 엔터티·관계, 커뮤니티 보고서**를 만드는 과정을 살펴봅니다. 자동 추출 결과가 사람이 만든 그래프와 같으리라고 가정하지 않습니다.

이 노트북에서 쓰는 자료와 파일 흐름은 다음과 같습니다.

```text
data/project/D1_owner.md … D6_decision.md      원본 Markdown 6개
        ↓ scripts/prepare_microsoft.py
runs/microsoft/input/*.pdf 또는 *.txt          GraphRAG에 넣는 사본
        ↓ graphrag index --method standard
runs/microsoft/output/*.parquet               추출·요약·연결 결과
        ↓ 이 노트북의 표·그림 확인 / graphrag query
출처를 확인한 답변
```

위 경로는 모두 **이 저장소 안**에 있습니다. `runs/`는 실행하면서 생기는 작업공간이므로 새로 내려받은 저장소에는 없습니다. 입력 준비는 API를 호출하지 않지만, 실제 인덱싱과 새 Local·Global 질의는 모델 API를 사용합니다. Microsoft의 [기본 인덱싱 흐름](https://microsoft.github.io/graphrag/index/default_dataflow/)과 [출력 형식](https://microsoft.github.io/graphrag/index/outputs/)을 기준으로 설명합니다.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
from session_demo.data import project_documents
from inspect_microsoft import load_tables, show
from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)

WORKSPACE = ROOT / "runs" / "microsoft"
local_cli = Path(sys.executable).parent / ("Scripts/graphrag.exe" if os.name == "nt" else "bin/graphrag")
GRAPHRAG_CLI = str(local_cli) if local_cli.is_file() else shutil.which("graphrag")
documents = project_documents(("D1", "D2", "D3", "D4", "D5", "D6"))
display(pd.DataFrame([{
    "ID": d.id, "원본 파일": d.path.relative_to(ROOT).as_posix(),
    "제목": d.title, "원문": d.text,
} for d in documents]))
print("원본 문서 폴더:", ROOT / "data" / "project")
print("GraphRAG 작업공간:", WORKSPACE)


## 1. 원본을 확인하고 GraphRAG 입력을 준비하기

**바로 위 코드 셀**은 `project_documents(...)`로 `data/project/`의 D1–D6 Markdown을 읽어 파일명·제목·본문을 표시합니다. `.env`의 설정은 읽지만 키 값은 출력하지 않습니다. `WORKSPACE`는 GraphRAG가 입력과 산출물을 보관할 `runs/microsoft/`입니다.

**아래 준비 코드 셀**에서 `RUN_PREPARE=True`로 설정하면 `scripts/prepare_microsoft.py`가 실행됩니다. 이 스크립트는 다음 일을 합니다.

1. `graphrag init`으로 `runs/microsoft/settings.yaml`을 만듭니다. 추출 대상에 사람·프로젝트·프로그램·팀 등을 지정합니다.
2. `INPUT_FORMAT="pdf"`이면 원본 Markdown 여섯 개를 각각 **텍스트가 들어 있는 PDF**로 만들어 `runs/microsoft/input/`에 둡니다. GraphRAG가 MarkItDown으로 읽도록 설정하고, 변환된 텍스트에 원문 문장이 남아 있는지 확인합니다. 이미지 스캔 PDF의 OCR을 수행하는 예시는 아닙니다. `"text"`이면 같은 내용을 UTF-8 `.txt`로 복사합니다.
3. 준비된 입력 파일 목록과 GraphRAG CLI 설치 여부를 출력합니다. 여기까지는 인덱싱이나 답변 생성이 아닙니다.

`RUN_PREPARE=False`는 기존 작업공간만 확인합니다. 작업공간이 이미 있으면 `True`여도 다시 만들지 않으므로, **`INPUT_FORMAT`을 바꾸는 것만으로 기존 PDF가 TXT로 바뀌지는 않습니다.** 처음 준비할 때 형식을 정하고, 한 작업공간에 두 형식을 섞지 않습니다. 입력 준비에도 `requirements-microsoft.txt`로 설치한 CLI가 필요합니다.


In [ ]:
RUN_PREPARE = False
INPUT_FORMAT = "pdf"  # 텍스트형 PDF 시연. 같은 WORKSPACE에 두 형식을 섞지 않습니다.
if RUN_PREPARE:
    if WORKSPACE.exists():
        print("기존 작업공간:", WORKSPACE)
    else:
        subprocess.run([sys.executable, str(ROOT / "scripts" / "prepare_microsoft.py"), "--format", INPUT_FORMAT], check=True)
else:
    print(f"준비 명령: python scripts/prepare_microsoft.py --format {INPUT_FORMAT}")
print("GraphRAG CLI 설치 여부:", bool(GRAPHRAG_CLI))
if WORKSPACE.exists():
    print("현재 입력 파일:", [p.name for p in sorted((WORKSPACE / "input").iterdir()) if p.is_file()])


## 2. 문서로부터 인덱스 만들기

아래 셀의 `RUN_PAID_INDEX=True`는 준비된 `runs/microsoft/input/`을 [GraphRAG의 standard 인덱싱](https://microsoft.github.io/graphrag/index/default_dataflow/)으로 처리합니다. 셀은 먼저 CLI·작업공간·프로젝트 루트 `.env`의 `OPENAI_API_KEY`를 확인합니다. `--dry-run --skip-validation`은 설정을 살펴보는 단계이며 모델 연결 검증을 건너뜁니다. 이어지는 **두 번째 `index` 명령**이 실제 API를 호출합니다.

실제 실행에서는 입력 문서를 텍스트 조각인 `TextUnit`으로 나누고, LLM이 엔터티와 관계를 추출·요약합니다. 관계 그래프에서 커뮤니티를 찾고 보고서를 만든 뒤 검색에 쓸 임베딩도 생성합니다. 그 결과가 `runs/microsoft/output/`의 Parquet 표와 설정된 벡터 저장소에 남습니다. **Neo4j에 저장하는 과정은 아닙니다.** 원문이 짧아도 모델 호출 비용과 실행 시간이 듭니다.

`RUN_PAID_INDEX=False`이면 이 셀은 건너뛰지만, 이전에 만들어 둔 산출물이 있다면 뒤의 확인 셀은 읽을 수 있습니다. 새 작업공간에서 아직 인덱싱하지 않았다면 뒤의 표와 그림은 나타나지 않습니다.


In [ ]:
RUN_PAID_INDEX = False
if RUN_PAID_INDEX:
    if not WORKSPACE.exists():
        raise FileNotFoundError("먼저 작업공간을 준비하세요.")
    if not GRAPHRAG_CLI:
        raise FileNotFoundError("GraphRAG CLI가 없습니다. requirements-microsoft.txt를 설치하세요.")
    if not os.getenv("OPENAI_API_KEY", "").strip():
        raise RuntimeError("프로젝트 루트 .env에 OPENAI_API_KEY가 없습니다.")
    subprocess.run([GRAPHRAG_CLI, "index", "--root", str(WORKSPACE), "--method", "standard", "--dry-run", "--skip-validation"], check=True)
    subprocess.run([GRAPHRAG_CLI, "index", "--root", str(WORKSPACE), "--method", "standard"], check=True)
else:
    print("실제 인덱싱은 건너뛰었습니다. 설정을 확인한 뒤 RUN_PAID_INDEX=True로 바꾸세요.")


## 3. 인덱싱 결과와 원문 출처 확인하기

**첫 번째 코드 셀**은 `runs/microsoft/output/relationships.parquet`가 있는지 확인합니다. 있으면 `load_tables(output)`이 실제 Parquet 파일을 읽고, `show(tables)`가 각 표의 행 수·열 이름과 앞부분을 표시합니다. 이 셀은 새 그래프를 만들거나 API를 호출하지 않습니다.

| 산출물 | 무엇이 들어 있나 |
| --- | --- |
| `documents` | 입력 문서와 연결된 TextUnit ID |
| `text_units` | 문서에서 나눈 텍스트 조각과 원문 문서 ID |
| `entities`, `relationships` | 자동 추출된 대상과 대상 사이의 설명·근거 TextUnit ID |
| `communities`, `community_reports` | 관계 그래프의 묶음과 생성된 묶음별 요약 |

**바로 다음 코드 셀**은 관계 표의 **첫 행 한 개**를 예로 들어 `관계 → text_unit_ids → TextUnit → document_id → 원문 문서`를 따라갑니다. 첫 행은 설명용 표본이지 정답 관계로 고른 것이 아닙니다. 화면의 `description`은 모델이 생성한 설명이므로 실제 문장과 대조해야 합니다. [공식 출력 스키마](https://microsoft.github.io/graphrag/index/outputs/)


In [ ]:
output = WORKSPACE / "output"
if (output / "relationships.parquet").exists():
    tables = load_tables(output)
    show(tables)
else:
    tables = {}
    print("아직 실제 산출물이 없습니다:", output)
    print("이 셀의 표는 인덱싱 후에만 나타납니다. 가상 표를 실제 결과처럼 표시하지 않습니다.")


In [ ]:
if {"relationships", "text_units"} <= tables.keys() and not tables["relationships"].empty:
    relation = tables["relationships"].iloc[0]
    units = tables["text_units"].set_index("id")
    documents_by_id = tables["documents"].set_index("id") if "documents" in tables else None
    print("추출 관계:", relation.get("source"), "→", relation.get("target"))
    print("설명:", relation.get("description"))
    for unit_id in list(dict.fromkeys(relation.get("text_unit_ids", [])))[:3]:
        if unit_id in units.index:
            row = units.loc[unit_id]
            doc_id = row.get("document_id")
            title = documents_by_id.loc[doc_id].get("title") if documents_by_id is not None and doc_id in documents_by_id.index else "(제목 미확인)"
            print("TextUnit:", unit_id, "| 문서:", title, "| 문서 ID:", doc_id)
            print(str(row.get("text", ""))[:700])
else:
    print("관계→TextUnit→원문 확인은 인덱싱 후 실행됩니다.")


## 4. 추출된 관계 일부를 그림으로 보기

아래 코드는 **실제 `relationships` 표의 앞 18개 행**에서 `source`와 `target`을 읽어 작은 NetworkX 그림을 만듭니다. 그림을 그리기 위한 별도의 Python 작업이며, Microsoft GraphRAG가 Neo4j에 그래프를 넣었다는 뜻은 아닙니다.

이 그림은 연결을 한눈에 보는 용도입니다. 코드가 **무방향 그래프**로 그리므로 원래 관계의 방향과 설명은 보존되지 않습니다. 두 점 사이에 선이 보여도 직접 협업이나 원문에서 확인된 관계라고 단정하지 마세요. 관계 표의 `description`과 `text_unit_ids`를 원문과 대조해야 합니다. 2번 노트북에서 사람이 지정한 `LEADS`·`PART_OF`·`REVIEWS` 타입이 이 결과에 그대로 붙는 것도 아닙니다.


In [ ]:
if "relationships" in tables and not tables["relationships"].empty:
    import networkx as nx
    import matplotlib.pyplot as plt
    import textwrap
    from session_demo.plot_fonts import apply_korean_font

    frame = tables["relationships"].head(18)
    graph = nx.from_pandas_edgelist(frame, source="source", target="target")
    fig, ax = plt.subplots(figsize=(15, 8))
    pos = nx.spring_layout(graph, seed=7, k=1.6, iterations=200)
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color="#dbead9", node_size=850, edgecolors="#89a688")
    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color="#789079", width=1.3, alpha=0.7)
    labels = {node: textwrap.fill(str(node), width=15, break_long_words=False) for node in graph}
    nx.draw_networkx_labels(graph, pos, labels=labels, ax=ax, font_size=8,
                            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 0.15})
    apply_korean_font(ax)
    ax.margins(0.13)
    ax.axis("off")
    fig.tight_layout()
    display(fig)
    plt.close(fig)
else:
    print("시각화할 실제 관계가 아직 없습니다.")


## 5. 자동 추출된 연결을 원문으로 되짚기

아래 코드는 `inspect_microsoft.py`의 `trace_entity_path`를 호출합니다. 이 함수는 **이미 생성된 관계 표를 따로 읽어** 오헤슬과 보안팀 사이의 짧은 연결을 찾고, 각 관계의 `text_unit_ids`와 `document_id`로 원문을 되짚습니다. Microsoft GraphRAG의 Local Search 답변 함수가 아니라 **산출물 점검용 코드**입니다.

이 점검에서는 관계를 무방향으로 이어 보므로 경로가 나타났다는 사실만으로 관계의 방향이나 뜻이 입증되지 않습니다. 원래 D1–D3에서는 오헤슬이 프로젝트를 총괄하고, 프로젝트가 프로그램에 속하며, 보안팀이 **프로그램**의 보안 검토를 맡습니다. 자동 추출 결과가 이 경로와 다르거나 경로가 없을 수도 있습니다. 각 행의 추출 설명을 옆에 나온 원문과 비교하고, **직접 협업했다는 기록은 원문에 없다는 점**을 확인합니다.


In [ ]:
from inspect_microsoft import trace_entity_path

if {"relationships", "text_units", "documents"} <= tables.keys():
    try:
        display(trace_entity_path(tables, "오헤슬", "보안팀"))
    except ValueError as exc:
        print("경로를 확인하지 못했습니다:", exc)
else:
    print("인덱싱 산출물을 먼저 준비하세요.")


## 6. 인덱스를 사용해 Local·Global 질문하기

앞의 표·그림은 **인덱스를 읽어 검사**한 결과입니다. 아래 셀은 `graphrag query` CLI로 그 인덱스에 실제 질문을 보내 **새 답변을 생성**하는 단계입니다. [공식 질의 설명](https://microsoft.github.io/graphrag/query/overview/)에 따르면 Local Search는 질문과 관련된 엔터티·관계·원문 조각을 중심으로, Global Search는 커뮤니티 보고서를 종합해 답합니다.

`RUN_PAID_QUERIES=True`이면 로컬 관계 질문과 전체 주제 질문을 차례로 실행해 답변을 노트북에 표시하고 `runs/microsoft/query_local_answer.md`, `query_global_answer.md`에도 저장합니다. API 호출 비용이 발생합니다. `False`이면 **새 질의를 보내지 않고**, 해당 파일이 있을 때만 이전 답변을 읽습니다. 새로 내려받은 저장소에는 `runs/`가 없어 이전 답변도 없습니다.

질의 결과의 인용 표시와 문장을 원문에 다시 대조합니다. 여섯 문서만으로 Global Search의 일반적인 성능을 평가할 수는 없습니다.


In [ ]:
RUN_PAID_QUERIES = False
if RUN_PAID_QUERIES:
    if not (output / "relationships.parquet").exists():
        raise FileNotFoundError("인덱싱 산출물이 먼저 필요합니다.")
    if not GRAPHRAG_CLI:
        raise FileNotFoundError("GraphRAG CLI가 없습니다. requirements-microsoft.txt를 설치하세요.")
    for method, question in [
        ("local", "오헤슬과 보안팀은 어떤 관계인가? 각 관계의 출처를 밝혀줘."),
        ("global", "이 자료 전체에서 출시와 보안 검토의 주요 주제는 무엇인가?"),
    ]:
        print("\n질의 방식:", method, "| 질문:", question)
        result = subprocess.run(
            [GRAPHRAG_CLI, "query", question, "--root", str(WORKSPACE), "--method", method],
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            env={**os.environ, "PYTHONIOENCODING": "utf-8"},
        )
        if result.returncode:
            print((result.stderr or result.stdout)[-2500:])
            raise RuntimeError(f"{method} 질의 실패. 상세 로그: {WORKSPACE / 'logs' / 'query.log'}")
        answer = result.stdout.strip()
        if not answer:
            raise RuntimeError(f"{method} 질의 응답이 비었습니다. 상세 로그: {WORKSPACE / 'logs' / 'query.log'}")
        display(Markdown(f"### {method.upper()} 답변\n\n{answer}"))
        answer_file = WORKSPACE / f"query_{method}_answer.md"
        answer_file.write_text(answer, encoding="utf-8")
        print("답변 저장:", answer_file)
else:
    print("새 API 질의를 건너뛰고 저장된 답변을 보여줍니다.")
    for method in ("local", "global"):
        answer_file = WORKSPACE / f"query_{method}_answer.md"
        if answer_file.exists():
            display(Markdown(f"### {method.upper()} 저장 답변\n\n{answer_file.read_text(encoding='utf-8')}"))
            print("읽은 파일:", answer_file)
        else:
            print(f"{method} 저장 답변이 없습니다. RUN_PAID_QUERIES=True로 한 번 질의하세요.")


### 저장된 답변을 읽을 때 확인할 것

앞 셀에서 `RUN_PAID_QUERIES=False`로 두면 저장된 답변이 있을 때만 이를 다시 보여줍니다. `True`이면 새로 생성한 답변을 표시하고 파일에도 저장합니다.

답변의 `[Data: ...]` 표시는 참조한 인덱스 자료의 위치를 가리킬 뿐 **문장 전체가 사실이라는 보증은 아닙니다.** D1–D3에는 직접 협업 기록이 없고, D5는 위험 등록부의 **문서 ID**이지 프로젝트 이름이 아닙니다. D6의 일정 변경 이유는 인증 절차 수정 확인이 더 필요하다는 것입니다. 생성 답변이 이를 다르게 말하면 원문과 근거 표를 다시 확인합니다.


**다음:** `04_llm_wiki.ipynb`에서는 검색 인덱스 대신 사람이 읽을 수 있는 Markdown 지식 페이지가 D6 이후 어떻게 갱신되는지 보여줍니다. 두 방식은 서로 결합할 수도 있습니다.
